# Block-Level Compressibility

This notebook asks what becomes predictable after observing the current reasoning block.

Unit of analysis: one block-summary pair.

Target: `high_token_compression`, created in `01_data_loading_engineering.ipynb` from the training-split block compression threshold.

Predictors: problem-derived features, metadata, the current block index, and current block length features.

Retrospective full-trace variables such as `n_blocks_in_trace` and `relative_block_position` are intentionally excluded. Those variables require knowing the full trace and do not belong in the online/current-block task.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sys.path.append(str(Path("..").resolve()))

from src.reasoning_compression.features import (
    count_share_table,
    latest_feature_build_dir,
    normalize_difficulty,
)

from src.reasoning_compression.modeling import (
    GRADIENT_BOOSTING_PARAM_GRID,
    RANDOM_FOREST_PARAM_GRID,
    random_forest_feature_importance,
    select_model_params,
)

In [2]:
FULL_BUILD_DIR = latest_feature_build_dir(Path("../data/full_feature_builds"))
TARGET = "high_token_compression"
SPLIT_COL = "model_split"
GROUP_COL = "trace_id"

df_blocks = pd.read_parquet(
    FULL_BUILD_DIR / "blocks_features_full_labeled.parquet"
)
df_blocks["difficulty"] = df_blocks["difficulty"].map(normalize_difficulty)

required_columns = {TARGET, SPLIT_COL, GROUP_COL}
missing_columns = required_columns.difference(df_blocks.columns)
if missing_columns:
    raise ValueError(
        "Block table is missing required columns: "
        f"{sorted(missing_columns)}. Re-run 01_data_loading_engineering.ipynb "
        "with RUN_FULL_BUILD = False."
    )

df_blocks.shape

(2013510, 23)

In [3]:
split_target_distribution = (
    df_blocks
    .groupby(SPLIT_COL)[TARGET]
    .value_counts(normalize=True)
    .rename("share")
    .reset_index()
    .assign(share_pct=lambda df: (df["share"] * 100).round(2))
    .drop(columns="share")
)

split_target_distribution

,model_split,high_token_compression,share_pct
0,test,0,75.03
1,test,1,24.97
2,train,0,75.00
3,train,1,25.00


**Block split check interpretation**

The block-level target distribution is aligned with the training-threshold design: the training split has exactly 25% high-compression blocks and the test split is nearly identical. The split is therefore suitable for comparing online block-level models without recomputing the target on test data.

## Feature Set

The block-level task excludes features that reveal future trace structure. `block_index` is kept because it is known at block `t`; `n_blocks_in_trace` and `relative_block_position` are excluded because they require knowing the final trace length.

In [4]:
numeric_features = [
    "block_index",
    "block_chars",
    "block_tokens",
    "problem_chars",
    "problem_tokens",
    "problem_math_symbol_share",
    "problem_question_mark_count",
]

binary_features = [
    "problem_has_multiple_choice",
    "problem_has_code_fence",
]

categorical_features = [
    "domain",
    "source",
    "difficulty",
]

feature_columns = numeric_features + binary_features + categorical_features

retrospective_features = {"n_blocks_in_trace", "relative_block_position"}
leaked_features = retrospective_features.intersection(feature_columns)
if leaked_features:
    raise ValueError(
        "Retrospective full-trace features are not allowed here: "
        f"{sorted(leaked_features)}"
    )

In [5]:
df_model = df_blocks[
    feature_columns + [TARGET, GROUP_COL, SPLIT_COL]
].dropna().copy()

X = df_model[feature_columns]
y = df_model[TARGET]
groups = df_model[GROUP_COL]
model_split = df_model[SPLIT_COL]

train_mask = model_split == "train"
test_mask = model_split == "test"

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]
groups_train = groups.loc[train_mask]
groups_test = groups.loc[test_mask]

X_train.shape, X_test.shape

((1611167, 12), (402343, 12))

In [6]:
len(set(groups_train).intersection(set(groups_test)))

0

## Modeling Utilities

Parameter grids, grouped validation tuning, and feature-importance helpers are imported from `src.reasoning_compression.modeling`. Preprocessing, fitting, and evaluation are defined directly with scikit-learn in this notebook.


In [7]:
rf_param_grid = RANDOM_FOREST_PARAM_GRID
gb_param_grid = GRADIENT_BOOSTING_PARAM_GRID

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("bin", "passthrough", binary_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ]
)

## Baseline

In [9]:
block_dummy_model = DummyClassifier(strategy="most_frequent")
block_dummy_model.fit(X_train, y_train)

block_dummy_pred = block_dummy_model.predict(X_test)
block_dummy_proba = block_dummy_model.predict_proba(X_test)[:, 1]
block_dummy_metrics = {
    "accuracy": accuracy_score(y_test, block_dummy_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, block_dummy_pred),
    "roc_auc": roc_auc_score(y_test, block_dummy_proba),
}
pd.DataFrame([block_dummy_metrics]).style.format("{:.4f}")


,accuracy,balanced_accuracy,roc_auc
0,0.7503,0.5000,0.5000


## Random Forest

In [22]:
block_rf_selection_results = select_model_params(
    model_name="Random forest",
    model_class=RandomForestClassifier,
    param_grid=rf_param_grid,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
)

In [21]:
display(
    block_rf_selection_results.style
    .set_properties(
        **{
            "white-space": "pre-wrap",
            "word-break": "break-word",
        }
    )
)

,model,params,n_tuning_rows,n_fit_rows,n_validation_rows,accuracy,balanced_accuracy,roc_auc
0,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf': 5, 'n_estimators': 300, 'n_jobs': -1, 'random_state': 42}",264465,211089,53376,0.732370,0.712275,0.796975
1,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf': 5, 'n_estimators': 150, 'n_jobs': -1, 'random_state': 42}",264465,211089,53376,0.731396,0.710813,0.796420
2,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf': 2, 'n_estimators': 300, 'n_jobs': -1, 'random_state': 42}",264465,211089,53376,0.750244,0.701929,0.791374
3,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf': 2, 'n_estimators': 150, 'n_jobs': -1, 'random_state': 42}",264465,211089,53376,0.750206,0.701497,0.790650


In [11]:
block_rf_params = block_rf_selection_results.loc[0, "params"]
block_rf_model = Pipeline(
    steps=[
        ("preprocess", clone(preprocessor)),
        ("model", RandomForestClassifier(**block_rf_params)),
    ]
)
block_rf_model.fit(X_train, y_train)

block_rf_pred = block_rf_model.predict(X_test)
block_rf_proba = block_rf_model.predict_proba(X_test)[:, 1]
block_rf_metrics = {
    "accuracy": accuracy_score(y_test, block_rf_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, block_rf_pred),
    "roc_auc": roc_auc_score(y_test, block_rf_proba),
}
pd.DataFrame([block_rf_metrics]).style.format("{:.4f}")


,accuracy,balanced_accuracy,roc_auc
0,0.7403,0.7208,0.8053


In [12]:
random_forest_feature_importance(block_rf_model).head(20).round(4)

,feature,importance
0,num__block_tokens,0.3492
1,num__block_chars,0.2993
2,num__problem_chars,0.0835
3,num__block_index,0.0765
4,num__problem_math_symbol_share,0.0761
5,num__problem_tokens,0.0751
6,num__problem_question_mark_count,0.0094
7,cat__domain_science,0.0068
8,cat__domain_math,0.0057
9,cat__source_stackexchange-physics,0.0044


**Block feature importance interpretation**

Current-block size dominates the random forest: `block_tokens` and `block_chars` are far more important than metadata. `block_index` and prompt-level variables still matter, but the main signal appears after observing the current reasoning block. This supports the conceptual shift from prompt-level compressibility to current-block compressibility.

## Gradient Boosting

In [20]:
block_gb_selection_results = select_model_params(
    model_name="Gradient boosting",
    model_class=HistGradientBoostingClassifier,
    param_grid=gb_param_grid,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
)


In [19]:
display(
    block_gb_selection_results.style
    .format({metric: "{:.4f}" for metric in ["accuracy", "balanced_accuracy", "roc_auc"]})
    .set_properties(
        **{
            "white-space": "pre-wrap",
            "word-break": "break-word",
        }
    )
)

,model,params,n_tuning_rows,n_fit_rows,n_validation_rows,accuracy,balanced_accuracy,roc_auc
0,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",264465,211089,53376,0.7935,0.6423,0.8008
1,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0.1, 'learning_rate': 0.05, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",264465,211089,53376,0.7934,0.6415,0.8007
2,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0.1, 'learning_rate': 0.1, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",264465,211089,53376,0.7933,0.6414,0.8006
3,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0.0, 'learning_rate': 0.1, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",264465,211089,53376,0.7936,0.6419,0.8006
4,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularization': 0.1, 'learning_rate': 0.05, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",264465,211089,53376,0.7141,0.7204,0.8006
5,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",264465,211089,53376,0.7140,0.7205,0.8005
6,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0.1, 'learning_rate': 0.1, 'max_iter': 150, 'max_leaf_nodes': 15, 'random_state': 42}",264465,211089,53376,0.7934,0.6428,0.8005
7,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_iter': 150, 'max_leaf_nodes': 15, 'random_state': 42}",264465,211089,53376,0.7928,0.6415,0.8004
8,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularization': 0.0, 'learning_rate': 0.1, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",264465,211089,53376,0.7136,0.7201,0.8004
9,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularization': 0.1, 'learning_rate': 0.1, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",264465,211089,53376,0.7139,0.7203,0.8004


In [14]:
block_gb_params = block_gb_selection_results.loc[0, "params"]
block_gb_model = Pipeline(
    steps=[
        ("preprocess", clone(preprocessor)),
        ("model", HistGradientBoostingClassifier(**block_gb_params)),
    ]
)
block_gb_model.fit(X_train, y_train)

block_gb_pred = block_gb_model.predict(X_test)
block_gb_proba = block_gb_model.predict_proba(X_test)[:, 1]
block_gb_metrics = {
    "accuracy": accuracy_score(y_test, block_gb_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, block_gb_pred),
    "roc_auc": roc_auc_score(y_test, block_gb_proba),
}
pd.DataFrame([block_gb_metrics]).style.format("{:.4f}")


,accuracy,balanced_accuracy,roc_auc
0,0.7950,0.6509,0.8076


## Model Comparison

In [ ]:
block_model_results = pd.DataFrame([
    {
        "task": "Block-level compressibility",
        "feature_set": "Problem + current block + metadata",
        "model": "Dummy",
        "selected_params": None,
        **block_dummy_metrics,
    },
    {
        "task": "Block-level compressibility",
        "feature_set": "Problem + current block + metadata",
        "model": "Random forest",
        "selected_params": block_rf_params,
        **block_rf_metrics,
    },
    {
        "task": "Block-level compressibility",
        "feature_set": "Problem + current block + metadata",
        "model": "Gradient boosting",
        "selected_params": block_gb_params,
        **block_gb_metrics,
    },
])

display(
    block_model_results.style
    .format({
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "roc_auc": "{:.4f}",
        "selected_params": lambda value: "None" if value is None else repr(value),
    })
    .set_properties(
        subset=["selected_params"],
        **{
            "white-space": "pre-wrap",
            "word-break": "break-word",
        }
    )
)


**Block model comparison interpretation**

The block-level models are much stronger than the prompt-level models, which is the expected result once the current reasoning block is available. Random forest and gradient boosting have very similar ROC AUC around 80-81%, but they behave differently at the default classification threshold. Random forest gives better balanced accuracy, while gradient boosting gives higher raw accuracy because it predicts the majority class more conservatively.

In [16]:
block_report_frames = []
for model_name, prediction in [
    ("Random forest", block_rf_pred),
    ("Gradient boosting", block_gb_pred),
]:
    report = classification_report(
        y_test,
        prediction,
        output_dict=True,
        zero_division=0,
    )
    report_frame = pd.DataFrame(report).T.reset_index(names="label")
    report_frame.insert(0, "model", model_name)
    block_report_frames.append(report_frame)

block_reports = pd.concat(block_report_frames, ignore_index=True)
block_reports.round(4)


,model,label,precision,recall,f1-score,support
0,Random forest,0,0.8777,0.7597,0.8145,301862.0000
1,Random forest,1,0.4858,0.6819,0.5674,100481.0000
2,Random forest,accuracy,0.7403,0.7403,0.7403,0.7403
3,Random forest,macro avg,0.6817,0.7208,0.6909,402343.0000
4,Random forest,weighted avg,0.7798,0.7403,0.7527,402343.0000
5,Gradient boosting,0,0.8158,0.9387,0.8729,301862.0000
6,Gradient boosting,1,0.6636,0.3631,0.4694,100481.0000
7,Gradient boosting,accuracy,0.7950,0.7950,0.7950,0.7950
8,Gradient boosting,macro avg,0.7397,0.6509,0.6712,402343.0000
9,Gradient boosting,weighted avg,0.7778,0.7950,0.7722,402343.0000


**Block classification report interpretation**

The classification report makes the threshold tradeoff explicit. Random forest recalls many more high-compression blocks, while gradient boosting has higher precision for class 1 but misses a larger share of true high-compression blocks. For the current-block task, random forest is preferable if the goal is to flag candidate high-compression blocks; gradient boosting is preferable only if higher precision and raw accuracy are prioritized.